ENSURE the robot says HSR starto before running this program. Release all emergency buttons.

In [1]:
import os
import hsrb_interface
import rospy
import sys
from hsrb_interface import geometry

In [2]:
os.environ['ROS_MASTER_URI'] = 'http://hsrb.local:11311'
os.environ['ROS_IP'] = '10.42.0.1'
print(os.environ['ROS_MASTER_URI'])
print(os.environ['ROS_IP'])

http://hsrb.local:11311
10.42.0.1


In [3]:
# Move timeout[s]
_MOVE_TIMEOUT=60.0
# Grasp force[N]
_GRASP_FORCE=0.2
# TF name of the OBJECT
_BOTTLE_TF='ar_marker/4'
_CANE_TF='ar_marker/6'
_TOOTHPASTE_TF='ar_marker/3'
# TF name of the gripper
_HAND_TF='hand_palm_link'

In [4]:
# Preparation for using the robot functions
robot = hsrb_interface.Robot()
omni_base = robot.get('omni_base')
whole_body = robot.get('whole_body')
gripper = robot.get('gripper')
tts = robot.get('default_tts')

In [ ]:
# Posture that 0.02[m] front and rotate -1.57 around z-axis of the bottle maker
bottle_to_hand = geometry.pose(z=-0.05, ek=-1.57)
bottle_to_cane = geometry.pose(z=-0.02, ek=-1.57)
bottle_to_toothpaste = geometry.pose(z=-0.02, ek=-1.57)

# Posture to move the hand 0.1[m] up
hand_up = geometry.pose(x=0.1)

# Posture to move the hand 0.5[m] back
hand_back = geometry.pose(z=-0.5)

In [ ]:
# Greet
whole_body.move_to_go()
# omni_base.go_abs(0.001512409973396251, 0.0008787339109116682, 0.000972665471566844, 300.0)
whole_body.move_to_neutral()
rospy.sleep(3.0)
whole_body.move_to_go()
tts.say('おはようございます')
rospy.sleep(3.0)

In [ ]:
# Transit to initial grasping posture
whole_body.move_to_neutral()
# Look at the hand after the transition
whole_body.looking_hand_constraint = True
omni_base.go_abs(0, 0, 0, 0.0)

In [ ]:
  #omni_base.go_rel(HUMAN FACING)
whole_body.move_to_go()
omni_base.go_abs(1.095651005508386, 0.880950433864387, 2.878201240685029, 300.0)
whole_body.move_to_neutral()
tts.say('元気ですか？気分はどうですか？何が必要ですか')
whole_body.move_to_joint_positions({'head_tilt_joint': 1.0,'head_tilt_joint': -0.5})
rospy.sleep(3.0)

In [ ]:
#position to bottle
try:
     # Transit to initial grasping posture
    whole_body.move_to_go()
    # omni_base.go_rel(0.0, 0.0, 3, 300.0)
    omni_base.go_abs(0.8948337616446661, 1.090646930554913, -0.00487550863654986, 300.0)
    whole_body.move_to_neutral()
    rospy.sleep(1.0)
    whole_body.move_to_go()
    omni_base.go_pose(geometry.pose(z=-1.0, ei=3.14, ej=-1.77), 100.0, ref_frame_id='ar_marker/4')
    tts.say('ボトル持って行きます')
    omni_base.pose
except:
    tts.say('たすけてください')
    rospy.logerr('fail to init')
    sys.exit()

In [ ]:
gripper.command(1.2)

In [ ]:
#grasp bottle
try:
    rospy.sleep(2.0)
    whole_body.move_to_neutral()
    # Look at the hand after the transition
    whole_body.looking_hand_constraint = True
    # Move the hand to front of the bottle
    whole_body.move_end_effector_pose(bottle_to_hand, _BOTTLE_TF)
    # Specify the force to grasp
    gripper.apply_force(_GRASP_FORCE)
    # Wait time for simulator's grasp hack. Not needed on actual robot
    rospy.sleep(2.0)
    # Move the hand up on end effector coordinate
    whole_body.move_end_effector_pose(hand_up, _HAND_TF)
    # Move the hand back on end effector coordinate
    whole_body.move_end_effector_pose(hand_back, _HAND_TF)
    # Transit to initial posture
    whole_body.move_to_neutral()
except:
    tts.say('たすけてください')
    rospy.logerr('fail to init')
    sys.exit()

In [ ]:
#release bottle
try:
    whole_body.move_to_go()
    omni_base.go_abs(1.095651005508386, 0.880950433864387, 2.878201240685029, 300.0)
    whole_body.move_to_neutral()
    tts.say('どうぞ')
    rospy.sleep(2.0)
    whole_body.move_to_neutral()
    #Look at the hand after the transition
    whole_body.looking_hand_constraint = True
    # Move the hand to front of the bottle
    gripper.command(1.2)
    #omni_base.pose
    #whole_body.move_end_effector_pose(hand_back, _HAND_TF)
    #whole_body.move_to_go()
except:
    tts.say('たすけてください')
    rospy.logerr('fail to init')
    sys.exit()

In [ ]:
whole_body.move_to_go()
omni_base.go_abs(1, 0, 0, 0.0)
omni_base.go_abs(0, 0, 0, 0.0)